# Day 10: End-to-End Orchestration, Observability & Verification

This notebook simulates a complete End-to-End pipeline run of a synthetic patient from raw HL7 JSON -> Bronze -> Tokenization -> Silver OMOP -> DQ -> Gold readmission metric.

In [ ]:
import pandas as pd
import duckdb
import hashlib
from datetime import datetime, timedelta

con = duckdb.connect(database=':memory:')
print("EHDIP Local Simulation Harness Ready.")

In [ ]:
print("\n=== STEP 1: INGESTION (RAW -> BRONZE) ===")
# Simulating HL7 Payload ingestion
raw_hl7_json = [
    {'msg_id': 'MSG-001', 'pid': 'PAT-999', 'ssn': '999-99-9999', 'admit_date': '2023-10-01'},
    {'msg_id': 'MSG-002', 'pid': 'PAT-999', 'ssn': '999-99-9999', 'admit_date': '2023-10-15'} # Readmission
]
bronze_df = pd.DataFrame(raw_hl7_json)
print("Bronze Data:\n", bronze_df)

print("\n=== STEP 2: TOKENIZATION & DE-IDENTIFICATION ===")
salt = "ehdip_salt"
def deid_ssn(ssn):
    return "DEID-" + hashlib.sha256(f"{ssn}{salt}".encode()).hexdigest()[:10]

bronze_df['ssn_deid'] = bronze_df['ssn'].apply(deid_ssn)
bronze_df['admit_date'] = pd.to_datetime(bronze_df['admit_date'])
print("De-identified Data:\n", bronze_df[['pid', 'ssn_deid', 'admit_date']])

print("\n=== STEP 3: OMOP SILVER STANDARDIZATION ===")
silver_omop = pd.DataFrame({
    'condition_occurrence_id': bronze_df['msg_id'],
    'person_id': bronze_df['pid'],
    'condition_start_date': bronze_df['admit_date']
})
print("Silver OMOP Data:\n", silver_omop)

print("\n=== STEP 4: DATA QUALITY & DLQ ROUTING ===")
# Add a bad record for demonstration
bad_record = pd.DataFrame({'condition_occurrence_id': ['MSG-003'], 'person_id': [None], 'condition_start_date': [pd.to_datetime('2023-11-01')]})
silver_omop = pd.concat([silver_omop, bad_record], ignore_index=True)

valid_data = silver_omop.dropna(subset=['person_id'])
dlq_data = silver_omop[silver_omop['person_id'].isna()]
print("Valid Data Routed to Silver:\n", valid_data)
print("\nInvalid Data Routed to DLQ:\n", dlq_data)

print("\n=== STEP 5 & 6: GOLD DBT MARTS (Readmissions) ===")
con.register('silver_omop', valid_data)
gold_sql = """
WITH readmission_calc AS (
    SELECT 
        a.person_id,
        a.condition_occurrence_id AS index_admission_id,
        b.condition_occurrence_id AS readmission_id
    FROM silver_omop a
    JOIN silver_omop b 
      ON a.person_id = b.person_id
      AND a.condition_occurrence_id != b.condition_occurrence_id
      AND b.condition_start_date > a.condition_start_date
      AND b.condition_start_date <= a.condition_start_date + INTERVAL 30 DAY
)
SELECT 
    person_id,
    COUNT(DISTINCT readmission_id) as total_30_day_readmissions
FROM readmission_calc
GROUP BY person_id
"""
gold_df = con.execute(gold_sql).fetchdf()
print("Gold 30-Day Readmissions Mart:\n", gold_df)

print("\n=== E2E PIPELINE RUN COMPLETE ===")
